# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para las celdas Markdown.

Material educativo sujeto a revisión y validación en Microsoft Fabric.

# NB_03 — AI_ANALYZE_SENTIMENT_PANDAS

## 1. Objetivos y requisitos
Analizar opiniones con etiquetas estándar y personalizadas sin Spark. Requiere un runtime Fabric compatible, AI Functions habilitadas y SynapseML para pandas. Revisar [requisitos y configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview). Los textos son ficticios y las dos llamadas consumen capacidad.

In [1]:
# Instalación opcional en runtime Python; ejecutar una vez si faltan las bibliotecas.
INSTALL = False
if INSTALL:
    import subprocess, sys
    from urllib.request import urlretrieve
    wheels = []
    for url, name in [
        ("https://aka.ms/fabric-aifunctions-whl", "synapseml_internal-latest-py3-none-any.whl"),
        ("https://aka.ms/fabric-synapseml-core-whl", "synapseml_core-latest-py3-none-any.whl")
    ]:
        urlretrieve(url, name)
        wheels.append(name)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *wheels])
# Reiniciar la sesión si el runtime lo requiere.

StatementMeta(, 522da651-4d74-4b55-a121-6c572027d03c, 3, Finished, Available, Finished, False)

## 2. Configuración
Modelo gpt-5-mini, concurrencia de laboratorio 10 y barra de progreso con estadísticas. [Configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration).

In [2]:
import pandas as pd
import synapse.ml.aifunc as aifunc
from datetime import datetime, timezone
from time import perf_counter
import uuid
MODEL_NAME = "gpt-5-mini"
CONCURRENCY = 10
aifunc.default_conf.model_deployment_name = MODEL_NAME
aifunc.default_conf.reasoning_effort = "low"
aifunc.default_conf.concurrency = CONCURRENCY
aifunc.default_conf.progress_bar_mode = "stats"

StatementMeta(, 522da651-4d74-4b55-a121-6c572027d03c, 4, Finished, Available, Finished, False)

## 3. Referencia oficial del método
[Microsoft Learn — ai.analyze_sentiment con pandas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/analyze-sentiment)

Sintaxis documentada (adaptada con etiquetas distintas):
```python
df["sentiment"] = df["input"].ai.analyze_sentiment()
df["sentiment"] = df["input"].ai.analyze_sentiment("happy", "angry", "indifferent")
```
| Parámetro | Tipo | Valor predeterminado |
|---|---|---|
| `labels` (posicionales) | str | positive, negative, mixed, neutral |
| `conf` | Conf | configuración global |
| `column_type` | str | texto; usar `path` para archivos |

Retorna una Series alineada con el índice original. Un sentimiento indeterminado puede retornar nulo; neutral no equivale a nulo. Las etiquetas personalizadas sustituyen las estándar.

## 4. Dataset ficticio
Mismas ocho opiniones y referencias humanas del NB_03 PySpark V2.

In [3]:
data = [
    (1, "Excelente atención, resolvieron mi problema rápidamente.", "positive"),
    (2, "El servicio fue pésimo y nadie respondió mi reclamo.", "negative"),
    (3, "El producto es excelente, pero la entrega fue terrible.", "mixed"),
    (4, "El pedido llegó el martes a las 10 de la mañana.", "neutral"),
    (5, "Me encantó la facilidad de uso.", "positive"),
    (6, "La aplicación falla constantemente, no la recomiendo.", "negative"),
    (7, "Buen precio, aunque la calidad me decepcionó.", "mixed"),
    (8, "La solicitud tiene el número 1234.", "neutral")
]
df = pd.DataFrame(data, columns=["id_opinion", "opinion", "sentimiento_esperado"])
display(df)

StatementMeta(, 522da651-4d74-4b55-a121-6c572027d03c, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b6bd87d8-01d9-4d97-9f11-c312269a728f)

## 5. Análisis estándar
Conservar la Series devuelta para consultar estadísticas antes de agregarla al DataFrame.

In [4]:
runs = []
def ejecutar(nombre, *labels):
    run_id = str(uuid.uuid4())
    started = datetime.now(timezone.utc).isoformat()
    t0 = perf_counter()
    resultado = df["opinion"].ai.analyze_sentiment(*labels)
    elapsed = perf_counter() - t0
    raw = resultado.ai.stats
    runs.append((nombre, run_id, started, elapsed, raw, resultado))
    return resultado

sentimiento = ejecutar("default")
display(df.assign(sentimiento=sentimiento))

StatementMeta(, 522da651-4d74-4b55-a121-6c572027d03c, 6, Finished, Available, Finished, False)

ai.analyze_sentiment: 100%|██████████| 8/8 [00:01<00:00,  6.00it/s, cached=0, in=507, out=527, CU-s=35.86]                             


SynapseWidget(Synapse.DataFrame, 3c5521f4-be41-4225-9648-c512db35b9ab)

## 6. Etiquetas personalizadas
Segunda llamada independiente; las categorías se adaptan al caso de negocio.

In [5]:
personalizado = ejecutar("custom", "satisfecho", "insatisfecho", "ambivalente", "sin opinión")
display(df.assign(sentimiento_personalizado=personalizado))

StatementMeta(, 522da651-4d74-4b55-a121-6c572027d03c, 7, Finished, Available, Finished, False)

ai.analyze_sentiment: 100%|██████████| 8/8 [00:01<00:00,  5.94it/s, cached=0, in=619, out=567, CU-s=38.64]                             


SynapseWidget(Synapse.DataFrame, f21fde64-25df-4d90-a6ac-4e2a3d50d6d5)

## 7. Validación
Comparar contra referencias humanas; revisar nulos y discrepancias. La coincidencia no garantiza calidad general.

In [6]:
mapa = {"positive":"satisfecho", "negative":"insatisfecho", "mixed":"ambivalente", "neutral":"sin opinión"}
resultado = df.assign(sentimiento=sentimiento, personalizado=personalizado)
resultado["coincide_default"] = sentimiento.eq(df.sentimiento_esperado).fillna(False)
resultado["coincide_custom"] = personalizado.eq(df.sentimiento_esperado.map(mapa)).fillna(False)
display(resultado)
for col in ["sentimiento", "personalizado"]:
    display(resultado[col].value_counts(dropna=False))
print("Coincidencia estándar:", resultado.coincide_default.mean())
print("Coincidencia personalizada:", resultado.coincide_custom.mean())

StatementMeta(, 522da651-4d74-4b55-a121-6c572027d03c, 8, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 0ff8e31c-2ce3-4d44-85f8-e3e9560e230a)

SynapseWidget(Synapse.DataFrame, 9e27c08a-0c79-4825-b55b-49cabb006803)

SynapseWidget(Synapse.DataFrame, 7bbcbdfc-f5f5-4fa8-8280-0e9f8671fbf1)

Coincidencia estándar: 1.0
Coincidencia personalizada: 1.0


## 8. Estadísticas y tokens
Consultar cada ejecución por separado. No reemplazar métricas ausentes por cero.

In [ ]:
def normalizar(raw):
    if isinstance(raw, pd.DataFrame) and len(raw)==1:
        return raw.iloc[0].to_dict()
    if isinstance(raw, pd.Series):
        return raw.to_dict()
    if isinstance(raw, dict):
        return raw
    raise RuntimeError("Revisar estructura de ai.stats")
for nombre, _, _, _, raw, _ in runs:
    print(nombre)
    display(raw)

## 9. Estimación de CU-segundos
Tarifas de referencia de la serie para gpt-5-mini: entrada 8,40; caché 0,84; salida 67,23 CU-s por mil tokens. Consultar [facturación oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing) para verificar vigencia.

$$CU_s=((I-C)8.40+C0.84+O67.23)/1000$$

El razonamiento es informativo; no se añade dos veces. La estimación no incluye otros consumos del runtime.

## 10. Auditoría individual y acumulada
Registrar ambas llamadas, tiempo y memoria aproximada de DataFrames (no memoria máxima del proceso).

In [ ]:
records = []
for nombre, run_id, started, elapsed, raw, resultado_run in runs:
    stats = normalizar(raw)
    def metric(k):
        v = stats.get(k)
        return None if v is None or pd.isna(v) else int(v)
    I,C,O = [metric(k) for k in ("input_tokens","cached_tokens","output_tokens")]
    cu = None
    if (stats.get("model") in (MODEL_NAME,"gpt-5-mini-2025-08-07") and stats.get("client_type")=="fabric_llm_endpoint" and all(v is not None for v in (I,C,O)) and 0<=C<=I and O>=0):
        cu = ((I-C)*8.40+C*0.84+O*67.23)/1000
    audit = dict(run_id=run_id, execution_timestamp_utc=started, notebook_name="NB_03_AI_ANALYZE_SENTIMENT_PANDAS", operation="ai.analyze_sentiment:"+nombre, model_configured=MODEL_NAME, model_reported=stats.get("model"), num_input_rows=len(df), client_type=stats.get("client_type"), estimated_cu_seconds=cu, elapsed_seconds=elapsed, input_bytes=int(df.memory_usage(deep=True).sum()), output_bytes=int(resultado_run.memory_usage(deep=True)), concurrency=CONCURRENCY)
    for key in ("num_successful","num_exceptions","num_unevaluated","num_harmful","input_tokens","cached_tokens","output_tokens","reasoning_tokens"):
        audit[key]=metric(key)
    audit["execution_status"]="SUCCESS" if audit["num_successful"]==len(df) else "INCOMPLETE"
    records.append(audit)
audit_df=pd.DataFrame(records)
display(audit_df)
print("CU-s acumulados:", audit_df.estimated_cu_seconds.sum(min_count=len(audit_df)))

## 11. Persistencia opcional en Delta
Requiere Lakehouse adjunto, deltalake y pyarrow. Persiste en Files, no registra una tabla SQL. Deshabilitada por defecto.

In [ ]:
SAVE_DELTA = False
if SAVE_DELTA:
    from deltalake import write_deltalake
    import pyarrow as pa
    schema = pa.schema([(k, pa.int64() if k.startswith("num_") or k.endswith("_tokens") or k in ("input_bytes", "output_bytes", "concurrency") else pa.float64() if k in ("estimated_cu_seconds", "elapsed_seconds") else pa.string()) for k in audit_df.columns])
    write_deltalake("/lakehouse/default/Files/audit_ai_functions_pandas", pa.Table.from_pandas(audit_df, schema=schema, preserve_index=False), mode="append")

## 12. Multimodalidad (opcional)
Usar archivos autorizados y revisar [requisitos multimodales](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview). Consumo independiente, fuera de la auditoría principal.

In [ ]:
RUN_MULTIMODAL = False
if RUN_MULTIMODAL:
    archivos = pd.Series(["/lakehouse/default/Files/ejemplo.pdf"])
    resultado_archivos = archivos.ai.analyze_sentiment(column_type="path")
    display(resultado_archivos)
    display(resultado_archivos.ai.stats)

## 13. Comparación y conclusiones
Pandas opera sobre Series con ejecución inmediata; PySpark devuelve un DataFrame distribuido con evaluación diferida. La concurrencia de solicitudes no equivale al paralelismo Spark. Las métricas de memoria representan objetos, no el pico del proceso.

Referencias: [método oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/analyze-sentiment), [configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration), [facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing).